# Phase 6 (summarization): one Phase 1 model, the main grid
Set `MODEL` and `FINISH_IST` in cell 2, then **Save Version -> Save & Run All** (GPU T4 x2, internet on, secrets
GH_TOKEN + HF_TOKEN, input checkpoints-models (+ moe-checkpoint for MoE)). One notebook per model.
Pipeline: mid-training on the training documents + summaries -> LoRA + LoRMA -> knowledge distillation (half-size
student, teacher = the better of LoRA / LoRMA; not for the encoder-decoder) -> QLoRA + adapter -> results table,
submission files, zip. Every run measures its speed and trains only what fits before its stage's end time.

In [ ]:
# 1. setup: code from GitHub
import os, subprocess, time

T0 = time.time()
from kaggle_secrets import UserSecretsClient

s = UserSecretsClient()
try:
    os.environ["HF_TOKEN"] = s.get_secret("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret: data download without it")
GH = s.get_secret("GH_TOKEN")
REPO = "/kaggle/working/cs781-capstone"
cmds = (
    [["git", "clone", "-q", f"https://{GH}@github.com/NaveenOLA13/cs781-capstone.git", REPO]]
    if not os.path.isdir(REPO)
    else [["git", "-C", REPO, "fetch", "-q"], ["git", "-C", REPO, "reset", "-q", "--hard", "origin/main"]]
)
for c in cmds:
    if subprocess.run(c, capture_output=True).returncode != 0:
        raise RuntimeError("git failed: check the GH_TOKEN secret is attached to this notebook")
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# 2. settings: GPU check, model, finishing time, plan
MODEL = "llama"  # "llama" | "decoder" | "moe" | "encdec"
FINISH_IST = "08:30"  # everything finished by this time (India time; earlier than now = tomorrow)

import torch

assert torch.cuda.is_available(), "NO GPU: right panel -> Session options -> Accelerator -> GPU T4 x2, then run again"
print(f"{torch.cuda.device_count()} GPU(s): {torch.cuda.get_device_name(0)}")
PLAN = {  # mid-training tokens (0 = none), batch per step, windows generated together, knowledge distillation
    "llama": dict(mid=30_000_000, batch=8, gen_batch=8, kd=True),
    "decoder": dict(mid=30_000_000, batch=8, gen_batch=8, kd=True),
    "moe": dict(mid=15_000_000, batch=4, gen_batch=4, kd=True),
    "encdec": dict(mid=0, batch=16, gen_batch=8, kd=False),
}[MODEL]
MAIN, MORE = ("lora", "lorma"), ("qlora", "adapter")
COMMON = dict(
    max_len=1024,
    epochs=4,
    batch_size=PLAN["batch"],
    grad_accum=2,  # MoE: smaller batches (ST-MoE)
    gen_batch=PLAN["gen_batch"],
    eval_docs=25,
    beams=2,
    no_repeat=3,
    log_every=200,
)
OUT = "/kaggle/working/summ_runs"

import sys, glob, json, shutil, datetime
from zoneinfo import ZoneInfo

sys.path.insert(0, f"{REPO}/phase6")
import p6_paths  # Phase 4 / 5 building blocks

IST = ZoneInfo("Asia/Kolkata")
_now = datetime.datetime.now(IST)
_h, _m = map(int, FINISH_IST.split(":"))
_fin = _now.replace(hour=_h, minute=_m, second=0, microsecond=0)
if _fin <= _now:
    _fin += datetime.timedelta(days=1)
FINISH = _fin.timestamp()
TRAIN_END = FINISH - 20 * 60  # the last 20 min: table, zip, saving the output


def minutes_left(t=None):
    return ((TRAIN_END if t is None else t) - time.time()) / 60


def ist(t):
    return datetime.datetime.fromtimestamp(t, IST).strftime("%d %b %H:%M")


assert minutes_left(FINISH) < 11.7 * 60, "FINISH_IST is more than 11.7 hours away (Kaggle stops a version at 12 h)"
assert minutes_left() > 120, f"only {minutes_left():.0f} min until {ist(TRAIN_END)}: set a later FINISH_IST"
print(f"now {_now:%d %b %H:%M} IST; training until {ist(TRAIN_END)}, everything finished by {ist(FINISH)} IST")

HINTS = {"llama": ["llama"], "decoder": ["decoder"], "encdec": ["encdec", "encoder_decoder", "enc_dec"], "moe": ["moe"]}


def find_ckpt(model):
    c = set(glob.glob("/kaggle/input/**/*.pt", recursive=True))
    c |= {os.path.dirname(p) for p in glob.glob("/kaggle/input/**/data.pkl", recursive=True)}
    hits = []
    for p in sorted(c):
        low = p.lower().replace("-", "_")
        if "last" in os.path.basename(low) or "/mid" in low or "summ_runs" in low or "lsi_runs" in low:
            continue
        if model == "decoder" and any(h in low for h in HINTS["encdec"]):
            continue
        if any(h in low for h in HINTS[model]):
            hits.append(p)
    hits.sort(key=lambda p: (not p.endswith(".pt"), len(p)))
    return hits[0] if hits else None


CKPT = find_ckpt(MODEL)
print("Phase 1 checkpoint:", CKPT)
assert CKPT, "attach the checkpoints-models dataset (and moe-checkpoint for MoE) and wait until it is ready"

In [ ]:
# 3. the data: sizes and lengths in our tokens, how much of a summary is copied, where its content comes from
import summ_data as SD
from cj_models import load_tokenizer
from rr_backbone import make_encode

data = SD.load_data(log=print)
print(SD.explore(data, make_encode(load_tokenizer())))
print("\nexample reference summary:\n", data["train"][0]["summary"][:800])

In [ ]:
# 4. self-tests on tiny stand-in models (~4 min); stops the notebook if something is broken
if not os.path.exists("/kaggle/working/tests6_passed"):
    r = subprocess.run(
        f"cd {REPO}/phase6 && python test_summ_data.py && python test_summ_train.py",
        shell=True,
        capture_output=True,
        text=True,
    )
    print(r.stdout[-5000:], r.stderr[-3000:])
    assert r.returncode == 0, "self-test failed: send the output above"
    open("/kaggle/working/tests6_passed", "w").write("ok")
print(f"tests passed; {minutes_left():.0f} min of training time left")

In [ ]:
# 5. mid-training: next-token prediction on the training documents + summaries (no held-out or test text)
if PLAN["mid"]:
    from summ_mid import mid_train

    mid_end = time.time() + min(45 * 60, 0.12 * (TRAIN_END - time.time()))
    print(f"mid-training until {ist(mid_end)} IST (at most {PLAN['mid'] / 1e6:.0f}M tokens)")
    START = mid_train(MODEL, path=CKPT, tokens=PLAN["mid"], batch_size=8 if MODEL == "moe" else 16, finish_by=mid_end)
else:
    START = CKPT  # the encoder-decoder is fine-tuned from its Phase 1 checkpoint
print("fine-tuning starts from", START, f"| {minutes_left():.0f} min of training time left")

In [ ]:
# 6. LoRA + LoRMA (one per GPU)
from summ_grid import run_grid, run_kd, report, best_run

share = 0.38 if PLAN["kd"] else 0.5
main_end = time.time() + share * (TRAIN_END - time.time())
print(f"{MAIN} until {ist(main_end)} IST")
try:
    run_grid(MODEL, techniques=MAIN, ckpt=START, out_dir=OUT, finish_by=main_end, **COMMON)
except Exception as e:
    print("main runs failed:", type(e).__name__, e)
print(f"{minutes_left():.0f} min of training time left")

In [ ]:
# 7. knowledge distillation: teacher = the better of LoRA / LoRMA on held-out ROUGE; student = the same mid-trained
#    model with half of its blocks (evenly spaced), trained with the teacher and, for comparison, without it
teacher = best_run(OUT, [f"{MODEL}_{t}" for t in MAIN])
if not PLAN["kd"]:
    print("knowledge distillation is not run for the encoder-decoder")
elif teacher is None:
    print("no finished teacher run: knowledge distillation skipped")
elif minutes_left() > 60:
    kd_end = time.time() + 0.45 * (TRAIN_END - time.time())
    print(f"teacher {teacher}; KD until {ist(kd_end)} IST")
    try:
        run_kd(
            teacher,
            student_layers=0.5,
            out_dir=OUT,
            finish_by=kd_end,
            lr_body=1e-4,
            epochs=COMMON["epochs"],
            log_every=200,
        )
    except Exception as e:
        print("KD failed:", type(e).__name__, e)
print(f"{minutes_left():.0f} min of training time left")

In [ ]:
# 8. QLoRA + adapter (one per GPU) until the end of the training time
if minutes_left() > 40:
    try:
        run_grid(MODEL, techniques=MORE, ckpt=START, out_dir=OUT, finish_by=TRAIN_END, **COMMON)
    except Exception as e:
        print("runs failed:", type(e).__name__, e)
else:
    print("not enough time left for", MORE)

In [ ]:
# 9. one table of everything + the submission files (best held-out first) + results zip (no weights)
import zipfile
from IPython.display import FileLink, display

try:
    report([OUT], out_path="/kaggle/working/all_results.txt")
except Exception as e:
    print("report failed:", type(e).__name__, e)
name = f"summ_results_{MODEL}.zip"
with zipfile.ZipFile(f"/kaggle/working/{name}", "w", zipfile.ZIP_DEFLATED) as z:
    for base in (OUT, "/kaggle/working/mid6"):
        for root, _, files in os.walk(base):
            for f in files:
                if not f.endswith(".pt"):
                    full = os.path.join(root, f)
                    z.write(full, os.path.relpath(full, "/kaggle/working"))
    if os.path.exists("/kaggle/working/all_results.txt"):
        z.write("/kaggle/working/all_results.txt", "all_results.txt")
shutil.rmtree("/kaggle/working/summ_cache", ignore_errors=True)
display(FileLink(name))
print(f"done at {ist(time.time())} IST")